# La búsqueda del techo — el ensayo que la corre

Este cuaderno **corre** la búsqueda de techos y nada más. No arma ninguna tabla ni conclusión — eso vive en `Benchmark_Search_Report_v1.ipynb`, que lee el registro que esto deja y lo presenta. Es la misma división que ya tienen la campaña y su informe, y por la misma razón: re-renderizar el informe cuesta segundos porque no vuelve a buscar nada.

Existía la mitad que dibuja y no la que corre: la única forma de pedir esta búsqueda era el paso `search-pilot` llamando a `harness.run_search()` por su cuenta, así que el artefacto que se abre y se lee nunca ejercitaba el programa que produce lo que muestra.

> **Corre el ENSAYO de la búsqueda, siempre, y no hay forma de pedirle otra cosa.**
>
> La búsqueda a escala completa escribe `ceilings.json`, que es el registro que gobierna toda campaña, y son unas nueve horas y media. Es una decisión con su propia autorización y no entra por acá. Este cuaderno escribe `ceilings.pilot.json`, que es otro archivo, y al que el registro completo le gana siempre que exista.
>
> Su respuesta no se cita. La rampa avanza con la fracción de entrenamiento transcurrida, así que a escala corta satura en la segunda época y todo techo se alcanza casi enseguida: lo que mediría es un paisaje en el que la campaña no entrena nunca. Lo que contesta es *«¿el programa corre?»*, que es lo único que un ensayo puede contestar.
>
> **Por eso la escala acá NO se deriva de `config.is_pilot_scale()`,** que es lo que hace el cuaderno de la campaña. Esa lectura mira `EPOCHS` y `SEEDS` — el tamaño de la CAMPAÑA —, y la búsqueda tiene su propia escala declarada aparte (`PILOT_SEARCH_EPOCHS`/`PILOT_SEARCH_TRIALS` contra `SEARCH_EPOCHS`/`SEARCH_TRIALS`). Derivarla acá haría que el tamaño de la campaña decidiera si este cuaderno lanza la corrida larga, que es exactamente la puerta que se cerró cuando `with_ceilings_in_force` lanzaba nueve horas y media desde una celda sin decir que las estaba lanzando.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
from MIL_CREDA_Benchmark import config, harness

# El ensayo, fijo, y con nombre. La misma respuesta elige las dos cosas --- a qué
# escala corre la búsqueda y a qué archivo escribe ---, así que escribir `True`
# en los dos lugares serían dos ortografías de una decisión, y la que quede vieja
# se lee igual de verde que la que no.
ES_ENSAYO = True
DESTINO = config.ceilings_record_for(ES_ENSAYO)

print("escala:", "ensayo" if ES_ENSAYO else "completa")
print("escribe en:", DESTINO)
print("registro en vigor ahora mismo:", config.ceilings_provenance()["source"])

## Lo que cuesta

El aforo declarado, y ningún pronóstico cronometrado. La campaña paga una corrida
real antes de comprometerse con la rejilla entera porque un estimado del costo es
más barato que el costo; acá el ensayo COMPLETO es del orden de esa corrida, así
que cronometrar una para pronosticarlo costaría una fracción visible de lo que
pronostica.

In [ ]:
aforo = config.search_sizing()
print(f"motor: {aforo['engine']}")
print(f"la búsqueda a su escala declarada: {aforo['trials']} trials de "
      f"{aforo['epochs']} épocas por (familia, transferencia) — "
      f"{aforo['runs']} corridas")
print(f"este ensayo: {config.PILOT_SEARCH_TRIALS} trials de "
      f"{config.PILOT_SEARCH_EPOCHS} épocas — "
      f"{config.PILOT_SEARCH_TRIALS * aforo['families'] * aforo['transfers']} "
      "corridas")

## La corrida

Se busca una vez. Si el registro ya existe se lee y no se vuelve a buscar: que el
registro exista significa que la búsqueda contestó, y sobrescribir una respuesta
porque alguien quería otra es el refinanciamiento silencioso que la negativa de la
campaña existe para impedir. Para volver a empezar hay que borrarlo a mano.

In [ ]:
# La biblioteca computa y este cuaderno orquesta: `run_search` arma la reducción
# a la escala que le corresponde a la búsqueda, despacha al motor que
# `config.SEARCH_ENGINE` declara y relee del disco lo que quedó escrito. Nada de
# eso se vuelve a escribir acá.
registro = harness.run_search(pilot=ES_ENSAYO)

print("registro escrito en:", DESTINO)
print("familias con techo:", ", ".join(sorted(registro)))
print()
print("las tablas y la conclusión las arma Benchmark_Search_Report_v1.ipynb")